# ReLLa-lite retrieved-history instruction dataset

Цель ноутбука - подготовить дополнительный instruction dataset для ReLLa-lite эксперимента.

Идея:
- обычный prompt использует исходную историю пользователя;
- ReLLa-lite prompt выбирает из истории пользователя только те игры, которые наиболее семантически похожи на target game;
- выбор делается отдельно среди liked и disliked игр;
- target game используется только для retrieval, а не как label.

Выходные файлы:

```text
data/final/instruction_temporal_retrieved_topk/train.jsonl
data/final/instruction_temporal_retrieved_topk/val.jsonl
data/final/instruction_temporal_retrieved_topk/test.jsonl
outputs/qwen_qlora/rella_lite/dataset/retrieved_instruction_summary.json
```

Этот notebook не обучает модель. Он только готовит новый dataset для будущего zero-shot / QLoRA / prompt-comparison эксперимента.


In [ ]:
import os, json, zipfile, ast
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
TOP_K_LIKED = 5
TOP_K_DISLIKED = 5

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("F:/Coursework")

INSTRUCTION_DIR = PROJECT_ROOT / "data" / "final" / "instruction_temporal"
TABULAR_DIR = PROJECT_ROOT / "data" / "final" / "tabular_temporal"
HISTORIES_PATH = PROJECT_ROOT / "data" / "processed" / "user_histories_mvp_v4_temporal.parquet"
SEMANTIC_DIR = PROJECT_ROOT / "outputs" / "semantic"
SEMANTIC_ZIP = SEMANTIC_DIR / "steam_semantic_outputs.zip"

OUT_INSTRUCTION_DIR = PROJECT_ROOT / "data" / "final" / "instruction_temporal_retrieved_topk"
OUT_DIR = PROJECT_ROOT / "outputs" / "qwen_qlora" / "rella_lite" / "dataset"
OUT_INSTRUCTION_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_INSTRUCTION_DIR:", OUT_INSTRUCTION_DIR)
print("OUT_DIR:", OUT_DIR)


## 1. Load semantic artifacts


In [ ]:
expected_semantic_files = [
    "item_embeddings.npy",
    "item_embedding_index.parquet",
    "item_embedding_texts.parquet",
]

if SEMANTIC_ZIP.exists():
    missing = [name for name in expected_semantic_files if not list(SEMANTIC_DIR.rglob(name))]
    if missing:
        with zipfile.ZipFile(SEMANTIC_ZIP, "r") as z:
            z.extractall(SEMANTIC_DIR)
        print("Semantic zip extracted.")
    else:
        print("Semantic files already extracted.")

def find_semantic_file(name):
    hits = list(SEMANTIC_DIR.rglob(name))
    if not hits:
        raise FileNotFoundError(f"Required semantic artifact was not found: {name}")
    return hits[0]

embeddings = np.load(find_semantic_file("item_embeddings.npy"))
item_index = pd.read_parquet(find_semantic_file("item_embedding_index.parquet"))
item_texts = pd.read_parquet(find_semantic_file("item_embedding_texts.parquet"))

if "app_id" not in item_index.columns:
    raise ValueError("item_embedding_index.parquet must contain 'app_id' column.")
if "embedding_row" not in item_index.columns:
    item_index = item_index.copy()
    item_index["embedding_row"] = np.arange(len(item_index))

app_to_row = dict(zip(item_index["app_id"].astype(int), item_index["embedding_row"].astype(int)))

if "embedding_text" not in item_texts.columns:
    text_col_candidates = [c for c in item_texts.columns if "text" in c.lower()]
    if not text_col_candidates:
        raise ValueError("item_embedding_texts.parquet must contain 'embedding_text' or another text column.")
    item_texts = item_texts.rename(columns={text_col_candidates[0]: "embedding_text"})

app_to_text = dict(zip(item_texts["app_id"].astype(int), item_texts["embedding_text"].fillna("").astype(str)))

print("embeddings:", embeddings.shape)
print("items:", len(app_to_row))


## 2. Load instruction, tabular and histories


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

splits = {}
for split in ["train", "val", "test"]:
    instr = load_jsonl(INSTRUCTION_DIR / f"{split}.jsonl")
    tab = pd.read_parquet(TABULAR_DIR / f"{split}_tabular.parquet")
    splits[split] = {"instruction": instr, "tabular": tab}
    print(split, "instruction:", instr.shape, "tabular:", tab.shape)

histories = pd.read_parquet(HISTORIES_PATH)
print("histories:", histories.shape)
print("history columns:", histories.columns.tolist())


## 3. Helper functions


In [ ]:
def parse_list_value(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            val = ast.literal_eval(s)
            if isinstance(val, (list, tuple, np.ndarray)):
                return list(val)
            return [val]
        except Exception:
            return [v.strip() for v in s.split(",") if v.strip()]
    return [x]

def to_int_app_ids(values):
    result = []
    for x in parse_list_value(values):
        try:
            if pd.isna(x):
                continue
            result.append(int(x))
        except Exception:
            continue
    return result

def to_bool_label(x):
    if isinstance(x, str):
        return x.strip().lower() in ["1", "true", "yes", "recommended", "positive"]
    return bool(x)

def get_col(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

def get_embedding(app_id):
    try:
        idx = app_to_row.get(int(app_id))
    except Exception:
        return None
    if idx is None:
        return None
    return embeddings[idx]

def cosine_to_target(target_emb, app_id):
    emb = get_embedding(app_id)
    if target_emb is None or emb is None:
        return -1.0
    return float(np.dot(target_emb, emb))

def clean_text_for_prompt(text, max_chars=350):
    text = " ".join(str(text).split())
    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + "..."
    return text

def describe_game(app_id):
    text = app_to_text.get(int(app_id), "")
    text = clean_text_for_prompt(text, max_chars=350)
    return f"[app_id={int(app_id)}] {text}" if text else f"[app_id={int(app_id)}]"

def extract_liked_disliked(row):
    cols = list(row.index)
    liked_col = get_col(cols, ["liked_app_ids", "history_liked_app_ids", "liked_history_app_ids"])
    disliked_col = get_col(cols, ["disliked_app_ids", "history_disliked_app_ids", "disliked_history_app_ids"])
    if liked_col or disliked_col:
        liked = to_int_app_ids(row.get(liked_col)) if liked_col else []
        disliked = to_int_app_ids(row.get(disliked_col)) if disliked_col else []
        return liked, disliked

    app_col = get_col(cols, ["history_app_ids", "history_apps", "history_items", "history_app_id"])
    lab_col = get_col(cols, ["history_labels", "history_is_recommended", "history_outputs", "history_label"])
    if app_col and lab_col:
        apps = parse_list_value(row.get(app_col))
        labels = parse_list_value(row.get(lab_col))
        liked, disliked = [], []
        for app, lab in zip(apps, labels):
            try:
                app = int(app)
            except Exception:
                continue
            if to_bool_label(lab):
                liked.append(app)
            else:
                disliked.append(app)
        return liked, disliked

    return [], []

def select_retrieved_history(target_app_id, liked, disliked, top_k_liked=5, top_k_disliked=5):
    target_emb = get_embedding(target_app_id)
    liked_ranked = sorted(
        [(app, cosine_to_target(target_emb, app)) for app in liked if app != target_app_id],
        key=lambda x: x[1],
        reverse=True
    )
    disliked_ranked = sorted(
        [(app, cosine_to_target(target_emb, app)) for app in disliked if app != target_app_id],
        key=lambda x: x[1],
        reverse=True
    )
    return liked_ranked[:top_k_liked], disliked_ranked[:top_k_disliked]


## 4. Prepare split dataframe with history columns


In [ ]:
def infer_target_col(df):
    for c in ["target_app_id", "target_item_id", "app_id"]:
        if c in df.columns:
            return c
    raise ValueError("Target app id column was not found in tabular data.")

def attach_histories(tab):
    if "sample_id" in tab.columns and "sample_id" in histories.columns:
        cols_to_add = [c for c in histories.columns if c not in tab.columns or c == "sample_id"]
        return tab.merge(histories[cols_to_add], on="sample_id", how="left")
    return tab.copy()

for split in ["train", "val", "test"]:
    tab = splits[split]["tabular"]
    tab_h = attach_histories(tab)
    splits[split]["tabular_with_history"] = tab_h
    print(split, tab_h.shape)


## 5. Build ReLLa-lite instruction examples


In [ ]:
RELLA_INSTRUCTION = (
    "You are a recommendation model. Predict whether the user will recommend the target Steam game. "
    "Use the semantically retrieved user history and answer only Yes or No."
)

def build_rella_input(target_app_id, liked_ranked, disliked_ranked):
    liked_lines = []
    for app, sim in liked_ranked:
        liked_lines.append(f"- similarity={sim:.4f}: {describe_game(app)}")

    disliked_lines = []
    for app, sim in disliked_ranked:
        disliked_lines.append(f"- similarity={sim:.4f}: {describe_game(app)}")

    liked_block = "\n".join(liked_lines) if liked_lines else "- No semantically relevant liked games were found."
    disliked_block = "\n".join(disliked_lines) if disliked_lines else "- No semantically relevant disliked games were found."
    target_block = describe_game(target_app_id)

    return (
        "User history was filtered by semantic similarity to the target game.\n\n"
        "Most relevant liked/recommended games:\n"
        f"{liked_block}\n\n"
        "Most relevant disliked/not recommended games:\n"
        f"{disliked_block}\n\n"
        "Target game:\n"
        f"{target_block}\n\n"
        "Question: will the user recommend the target game? Answer only Yes or No."
    )

def build_retrieved_split(split_name):
    instr = splits[split_name]["instruction"].reset_index(drop=True)
    tab_h = splits[split_name]["tabular_with_history"].reset_index(drop=True)
    target_col = infer_target_col(tab_h)

    if len(instr) != len(tab_h):
        raise ValueError(f"{split_name}: instruction and tabular splits have different row counts.")

    rows = []
    stats = {
        "split": split_name,
        "rows": int(len(instr)),
        "with_liked_selected": 0,
        "with_disliked_selected": 0,
        "avg_liked_selected": 0.0,
        "avg_disliked_selected": 0.0,
        "target_embedding_missing": 0,
    }
    liked_counts = []
    disliked_counts = []

    for i in range(len(instr)):
        source = instr.iloc[i].to_dict()
        tab_row = tab_h.iloc[i]
        target_app_id = int(tab_row[target_col])
        liked, disliked = extract_liked_disliked(tab_row)
        liked_ranked, disliked_ranked = select_retrieved_history(
            target_app_id,
            liked,
            disliked,
            TOP_K_LIKED,
            TOP_K_DISLIKED
        )

        if get_embedding(target_app_id) is None:
            stats["target_embedding_missing"] += 1
        if liked_ranked:
            stats["with_liked_selected"] += 1
        if disliked_ranked:
            stats["with_disliked_selected"] += 1

        liked_counts.append(len(liked_ranked))
        disliked_counts.append(len(disliked_ranked))

        source["instruction"] = RELLA_INSTRUCTION
        source["input"] = build_rella_input(target_app_id, liked_ranked, disliked_ranked)
        source["retrieval_top_k_liked"] = TOP_K_LIKED
        source["retrieval_top_k_disliked"] = TOP_K_DISLIKED
        if "sample_id" in tab_h.columns:
            source["sample_id"] = tab_h.iloc[i]["sample_id"]
        rows.append(source)

        if (i + 1) % 20000 == 0:
            print(split_name, "processed", i + 1)

    stats["avg_liked_selected"] = float(np.mean(liked_counts)) if liked_counts else 0.0
    stats["avg_disliked_selected"] = float(np.mean(disliked_counts)) if disliked_counts else 0.0
    return pd.DataFrame(rows), stats


## 6. Save retrieved instruction JSONL


In [ ]:
def save_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in df.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "task": "rella_lite_retrieved_instruction_dataset",
    "top_k_liked": TOP_K_LIKED,
    "top_k_disliked": TOP_K_DISLIKED,
    "source_instruction_dir": str(INSTRUCTION_DIR),
    "output_instruction_dir": str(OUT_INSTRUCTION_DIR),
    "splits": [],
}

for split in ["train", "val", "test"]:
    retrieved_df, stats = build_retrieved_split(split)
    out_path = OUT_INSTRUCTION_DIR / f"{split}.jsonl"
    save_jsonl(retrieved_df, out_path)
    summary["splits"].append(stats)
    print(split, stats, "saved:", out_path)

with open(OUT_DIR / "retrieved_instruction_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("saved summary:", OUT_DIR / "retrieved_instruction_summary.json")


## 7. Quick preview


In [ ]:
preview_path = OUT_INSTRUCTION_DIR / "test.jsonl"
with open(preview_path, "r", encoding="utf-8") as f:
    first = json.loads(next(f))

print("instruction:")
print(first.get("instruction", "")[:1000])
print("\ninput:")
print(first.get("input", "")[:2500])
print("\noutput:", first.get("output"))
